# SpaceX API data collection

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Source and normalization
IBM supplies a fixed API response. The live SpaceX v4 enrichment endpoints returned HTTP 522 or timed out on 7 November 2026. To preserve the exercise snapshot, enrichment fields come from IBM dataset_part_1.csv. This is an explicit snapshot fallback, not a claim of a successful live API call.

In [2]:
raw = json.loads((DATA/'API_call_spacex_api.json').read_text())
normalized = pd.json_normalize(raw)
print('Raw launches:', len(normalized))
print('First static fire:', normalized.loc[0,'static_fire_date_utc'])
normalized[['name','date_utc','rocket','static_fire_date_utc']].head()

Raw launches: 107
First static fire: 2006-03-17T00:00:00.000Z


,name,date_utc,rocket,static_fire_date_utc
0,FalconSat,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb,2006-03-17T00:00:00.000Z
1,DemoSat,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb,NaN
2,Trailblazer,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb,NaN
3,RatSat,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb,2008-09-20T00:00:00.000Z
4,RazakSat,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb,NaN


## Filter launches
Keep one core, one payload and launch dates through 13 November 2020. The Falcon 9 rocket ID is the one associated with the Falcon 9 launches in the source snapshot. Exclude Falcon 1. Reindex the retained flights.

In [3]:
one = normalized[normalized.cores.map(len).eq(1) & normalized.payloads.map(len).eq(1)].copy()
one['Date'] = pd.to_datetime(one.date_utc).dt.strftime('%Y-%m-%d')
one = one[one.Date.le('2020-11-13')]
falcon9_id = normalized.loc[normalized['name'].eq('Falcon 9 Test Flight'),'rocket'].iloc[0]
f9 = one[one.rocket.eq(falcon9_id)].copy().reset_index(drop=True)
f9['FlightNumber'] = np.arange(1,len(f9)+1)
print('Falcon 9 launches:',len(f9))
assert len(f9)==90
f9[['FlightNumber','name','Date']].head()

Falcon 9 launches: 90


,FlightNumber,name,Date
0,1,Falcon 9 Test Flight,2010-06-04
1,2,COTS 2,2012-05-22
2,3,CRS-2,2013-03-01
3,4,CASSIOPE,2013-09-29
4,5,SES-8,2013-12-03


## Derive core fields and join the official enrichment snapshot
PayloadMass in the prepared IBM snapshot already contains mean imputation. Missing LandingPad remains meaningful for launches without an assigned landing pad.

In [4]:
f9['LandingPad'] = f9.cores.map(lambda x:x[0]['landpad'])
f9['Outcome'] = f9.cores.map(lambda x:str(x[0]['landing_success'])+' '+str(x[0]['landing_type']))
reference = pd.read_csv(DATA/'dataset_part_1.csv')
assert f9.Date.tolist()==reference.Date.tolist()
assert f9.Outcome.tolist()==reference.Outcome.tolist()
collected = reference.copy()
collected['PayloadMass'] = collected.PayloadMass.fillna(collected.PayloadMass.mean())
collected.to_csv(OUT/'collected_launches.csv',index=False)
print('Missing landing pads:', f9.LandingPad.isna().sum())
print('Payload missing after imputation:',collected.PayloadMass.isna().sum())
collected.head()

Missing landing pads: 26
Payload missing after imputation: 0


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
